
# Assignment 3: Dialogue Summarization with Seq2Seq LSTM and Transformers

In this assignment you will:
- Prepare the SAMSum dataset for dialogue summarization.
- Implement a word-level Seq2Seq LSTM.
- Fine-tune GPT-2 and T5-small on the same task.
- Compare ROUGE + token-overlap metrics.

Complete each TODO cell; check `assignment3_solution.ipynb` if you need a reference.


In [ ]:
!pip install transformers datasets evaluate rouge_score
import os
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    DataCollatorForLanguageModeling,
    DataCollatorForSeq2Seq,
    Trainer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainingArguments,
)

import evaluate

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set random seed for reproducibility
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=01cdbe654571ad828ee4611a1e701fca52a9cf366d4f2962fab03418247cc9a9
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


device(type='cuda')

## 1. Dataset: SAMSum (dialogue summarization)
We use `knkarthick/samsum` from Hugging Face:
- Each example has:
- `dialogue` - multi-turn chat-style conversation.
- `summary` - a human-written abstractive summary.
We **subsample** to keep runtime reasonable.

In [ ]:
# === TODO ===
# Use datasets.load_dataset("knkarthick/samsum") to bring the data in, print the dataset object,
# and inspect the first training dialogue + summary so you know the raw format you will parse later.
raw_datasets = load_dataset("knkarthick/samsum")
print(raw_datasets)
print(raw_datasets["train"][0]["dialogue"])
print(raw_datasets["train"][0]["summary"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)
Amanda baked cookies and will bring Jerry some tomorrow.


In [ ]:
# === TODO ===
# To keep runtime manageable, slice each split down (e.g., 800 train / 100 validation / 100 test)
# using select(range(n)). Afterward, print the split lengths and one example pair for sanity.
train_small = raw_datasets["train"].select(range(800))
val_small = raw_datasets["validation"].select(range(100))
test_small = raw_datasets["test"].select(range(100))
print(len(train_small), len(val_small), len(test_small))
print(train_small[0]["dialogue"])
print("--- SUMMARY ---")
print(train_small[0]["summary"])

800 100 100
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)
--- SUMMARY ---
Amanda baked cookies and will bring Jerry some tomorrow.


## 2. LSTM seq2seq summarizer (word-level)

We will:
1. Build a **word-level vocabulary** from training dialogues + summaries.
2. Encode text as integer sequences.
3. Implement **encoder-decoder LSTM** with teacher forcing.
4. Train with our own loop.
5. Decode with greedy search and evaluate with ROUGE + token overlap.
We simplify vocabulary to avoid `<unk>` everywhere by keeping **all tokens** from the training split (within a max size).

In [ ]:
# === TODO ===

def simple_tokenize(text: str):
    """Lowercase the string, strip whitespace, and split on spaces."""
    return text.lower().strip().split()

special_tokens = ["<pad>", "<sos>", "<eos>", "<unk>"]

def build_vocab(dataset, max_size=8000, min_freq=1):
    """Count tokens across train dialogues + summaries, keep frequent ones, return itos/stoi."""
    token_counts = Counter()
    for example in dataset:
        token_counts.update(simple_tokenize(example["dialogue"]))
        token_counts.update(simple_tokenize(example["summary"]))

    # Add special tokens first
    itos = list(special_tokens)
    # Add other tokens based on frequency, respecting max_size and min_freq
    for token, count in token_counts.most_common():
        if count >= min_freq and token not in itos:
            itos.append(token)
        if len(itos) >= max_size:
            break

    stoi = {token: i for i, token in enumerate(itos)}
    return itos, stoi

itos, stoi = build_vocab(train_small) # Call build_vocab with your training data

vocab_size = len(itos)
pad_idx = stoi["<pad>"]
sos_idx = stoi["<sos>"]
eos_idx = stoi["<eos>"]
unk_idx = stoi["<unk>"]
print("Vocab size:", vocab_size)

Vocab size: 8000


### 2.2 Encode one example into ids

In [ ]:
def tokens_to_ids(tokens, stoi_dict, unk_index):
    """Map each token to its integer id, falling back to unk_index."""
    return [stoi_dict.get(token, unk_index) for token in tokens]


def encode_example(dialogue: str, summary: str, stoi_dict):
    """Tokenize both fields, convert to ids, and wrap targets with <sos>/<eos>."""
    src_tokens = simple_tokenize(dialogue)
    src_ids = tokens_to_ids(src_tokens, stoi_dict, unk_idx)

    tgt_tokens = simple_tokenize(summary)
    tgt_ids = [sos_idx] + tokens_to_ids(tgt_tokens, stoi_dict, unk_idx) + [eos_idx]

    return src_ids, tgt_ids

src_ids, tgt_ids = encode_example(train_small[0]["dialogue"], train_small[0]["summary"], stoi)
print(len(src_ids), len(tgt_ids))
print(src_ids[:20])
print(tgt_ids[:20])

16 11
[868, 4, 3699, 5635, 29, 7, 61, 3700, 689, 646, 868, 47, 228, 7, 243, 647]
[1, 2861, 3699, 2329, 9, 15, 228, 1507, 45, 400, 2]


### 2.3 PyTorch Dataset and DataLoader
We need:
- A `Dataset` that returns `input_ids` (dialogue) and `labels` (summary).
- A `collate_fn` that:
- Pads sequences in each batch.
- Builds a source mask (1 for real tokens, 0 for pad).

In [ ]:
# === TODO ===
import torch.nn.utils.rnn as rnn_utils

class SummarizationSeq2SeqDataset(Dataset):
    """Store the HF subset and encode on the fly."""
    def __init__(self, hf_dataset, stoi_dict):
        self.hf_dataset = hf_dataset
        self.stoi_dict = stoi_dict

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        src_ids, tgt_ids = encode_example(example["dialogue"], example["summary"], self.stoi_dict)
        return {"input_ids": src_ids, "labels": tgt_ids}


def collate_fn(batch):
    """Pad variable-length sequences to the max in batch, build src_mask, return tensors."""
    # Separate input_ids and labels from the batch
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]

    # Pad sequences
    padded_src = rnn_utils.pad_sequence(input_ids, batch_first=True, padding_value=pad_idx)
    padded_tgt = rnn_utils.pad_sequence(labels, batch_first=True, padding_value=pad_idx)

    # Build src_mask: 1 for real tokens, 0 for pad tokens
    src_mask = (padded_src != pad_idx).long()

    return {
        "input_ids": padded_src,
        "labels": padded_tgt,
        "src_mask": src_mask,
    }

train_dataset_lstm = SummarizationSeq2SeqDataset(train_small, stoi)
val_dataset_lstm = SummarizationSeq2SeqDataset(val_small, stoi)

train_loader_lstm = DataLoader(train_dataset_lstm, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader_lstm = DataLoader(val_dataset_lstm, batch_size=32, shuffle=False, collate_fn=collate_fn)

batch = next(iter(train_loader_lstm))
batch["input_ids"].shape, batch["labels"].shape, batch["src_mask"].shape

(torch.Size([32, 290]), torch.Size([32, 51]), torch.Size([32, 290]))

### 2.4 LSTM encoder-decoder model
- Encoder:
- Embedding + LSTM over the dialogue.
- Returns final hidden state `(h, c)`.
- Decoder:
- Embedding + LSTM + linear projection to vocab.
- During training: teacher forcing with the full target sequence.
- During generation: greedy decoding from `<sos>` until `<eos>` or `max_len`.

In [ ]:
# === TODO === Seq2Seq LSTM architecture

class EncoderLSTM(nn.Module):
    """Embedding + batch-first LSTM returning (outputs, (h, c))."""
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        # Set batch_first=True for convenience with DataLoader and collate_fn
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, input_ids, src_mask):
        # input_ids: (batch_size, seq_len)
        # src_mask: (batch_size, seq_len)

        embedded = self.embedding(input_ids)  # (batch_size, seq_len, embed_dim)

        # Get actual lengths of sequences from the mask
        lengths = src_mask.sum(dim=1).cpu()

        # Pack padded sequences for efficient LSTM processing
        # enforce_sorted=False is important if lengths are not sorted
        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )

        # Pass through LSTM
        # outputs are packed, hidden and cell are (num_layers * num_directions, batch_size, hidden_size)
        _, (hidden, cell) = self.lstm(packed_embedded)

        # Return the final hidden and cell states (usually from the last layer if multi-layer)
        # For single-layer, num_layers=1, so we can directly return hidden and cell.
        return hidden, cell


class DecoderLSTM(nn.Module):
    """Embedding + LSTM + linear projection to vocab size."""
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids, hidden, cell):
        # input_ids: (batch_size, 1) - single token at a time
        # hidden: (1, batch_size, hidden_dim)
        # cell: (1, batch_size, hidden_dim)

        embedded = self.embedding(input_ids)  # (batch_size, 1, embed_dim)

        # Pass through LSTM with current hidden and cell states
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: (batch_size, 1, hidden_dim) since seq_len=1

        prediction = self.fc_out(output.squeeze(1))  # (batch_size, vocab_size)

        return prediction, hidden, cell


class Seq2SeqLSTM(nn.Module):
    """Tie encoder/decoder together; implement forward + greedy generate."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx):
        super().__init__()
        self.encoder = EncoderLSTM(vocab_size, embed_dim, hidden_dim)
        self.decoder = DecoderLSTM(vocab_size, embed_dim, hidden_dim)
        self.pad_idx = pad_idx
        self.vocab_size = vocab_size

    def forward(self, input_ids, labels, src_mask, teacher_forcing_ratio=0.5):
        # input_ids: (batch_size, src_seq_len)
        # labels: (batch_size, tgt_seq_len)
        # src_mask: (batch_size, src_seq_len)

        batch_size = labels.shape[0]
        tgt_len = labels.shape[1]

        # Encode the source sequence
        encoder_hidden, encoder_cell = self.encoder(input_ids, src_mask)

        # Initialize decoder inputs with <sos> tokens
        decoder_input = labels[:, 0].unsqueeze(1)  # (batch_size, 1) - starting with <sos>

        decoder_hidden = encoder_hidden
        decoder_cell = encoder_cell

        outputs = torch.zeros(batch_size, tgt_len, self.vocab_size).to(labels.device)

        for t in range(1, tgt_len): # Start from 1 because <sos> is already processed
            decoder_output, decoder_hidden, decoder_cell = self.decoder(decoder_input, decoder_hidden, decoder_cell)
            outputs[:, t, :] = decoder_output

            # Decide whether to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio

            # Get the best predicted token
            top1 = decoder_output.argmax(1)

            # Use ground truth next token if teacher forcing, else use predicted token
            decoder_input = labels[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)

        return outputs

    def generate(self, input_ids, src_mask, max_len=50):
        # input_ids: (batch_size, src_seq_len)
        # src_mask: (batch_size, src_seq_len)

        batch_size = input_ids.shape[0]

        encoder_hidden, encoder_cell = self.encoder(input_ids, src_mask)

        decoder_input = torch.tensor([sos_idx] * batch_size, device=input_ids.device).unsqueeze(1)

        decoder_hidden = encoder_hidden
        decoder_cell = encoder_cell

        generated_ids = []

        for t in range(max_len):
            decoder_output, decoder_hidden, decoder_cell = self.decoder(decoder_input, decoder_hidden, decoder_cell)

            top1 = decoder_output.argmax(1)

            generated_ids.append(top1)

            decoder_input = top1.unsqueeze(1) # Feed prediction back to decoder

            # Stop if all sequences have generated <eos>
            if (top1 == eos_idx).all():
                break

        return torch.stack(generated_ids, dim=1)


model_lstm = Seq2SeqLSTM(vocab_size, embed_dim=128, hidden_dim=256, pad_idx=pad_idx).to(device)

### 2.5 Training loop for LSTM
- Loss: cross-entropy, ignoring `<pad>`.
- Gradient clipping to avoid exploding gradients.
- We run a few epochs only. You can increase later for better summaries.

In [ ]:
# === TODO === Training utilities for the LSTM

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)

def train_epoch(model, dataloader):
    model.train()
    total_loss = 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        src_mask = batch["src_mask"].to(device)

        optimizer.zero_grad()

        # Forward pass
        output = model(input_ids, labels, src_mask)

        # The labels are shifted inside the model for teacher forcing.
        # We need to reshape the output and labels for CrossEntropyLoss.
        # output shape: (batch_size, tgt_len, vocab_size)
        # labels shape: (batch_size, tgt_len)
        # We want to predict from the second token onwards (the actual summary tokens).
        # The first token in `labels` is <sos>, which is used to start decoding.
        # The corresponding prediction for the first token will be ignored by `ignore_index` if it's pad_idx.
        # So we align labels with outputs.
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim) # Skip the first token (corresponds to <sos> input)
        labels = labels[:, 1:].reshape(-1) # Skip the first label (corresponds to <sos>)

        loss = criterion(output, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate_loss(model, dataloader):
    """Validation pass without gradient updates."""
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            src_mask = batch["src_mask"].to(device)

            output = model(input_ids, labels, src_mask, teacher_forcing_ratio=0.0) # No teacher forcing for eval

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            labels = labels[:, 1:].reshape(-1)

            loss = criterion(output, labels)
            total_loss += loss.item()

    return total_loss / len(dataloader)


num_epochs = 3
for epoch in range(1, num_epochs + 1):
    train_loss = train_epoch(model_lstm, train_loader_lstm)
    val_loss = evaluate_loss(model_lstm, val_loader_lstm)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

Epoch 1: train_loss=7.8395, val_loss=5.6807
Epoch 2: train_loss=6.3819, val_loss=5.6674
Epoch 3: train_loss=6.2287, val_loss=5.6372


### 2.6 Decoding and evaluation for LSTM
We define:
- `ids_to_text`: turns the generated id sequence into readable text.
- `token_overlap`: simple Jaccard similarity between prediction and reference tokens.
- `evaluate_lstm_rouge`: runs the model on a subset and computes ROUGE + average token overlap.

In [ ]:
# === TODO === Evaluation helpers for the LSTM

def ids_to_text(ids):
    """Convert generated ids back to tokens, dropping pad/sos and stopping at eos."""
    tokens = []
    for idx in ids:
        idx = idx.item() if isinstance(idx, torch.Tensor) else idx
        if idx == eos_idx or idx == pad_idx:
            break
        if idx == sos_idx:
            continue
        tokens.append(itos[idx])

    return " ".join(tokens)

def token_overlap(pred: str, ref: str):
    """Jaccard similarity over token sets is sufficient here."""
    p_tokens = set(simple_tokenize(pred))
    r_tokens = set(simple_tokenize(ref))
    if not p_tokens and not r_tokens:
        return 1.0 # Both empty, perfect overlap
    if not p_tokens or not r_tokens:
        return 0.0 # One is empty, no overlap
    return len(p_tokens.intersection(r_tokens)) / len(p_tokens.union(r_tokens))

rouge = evaluate.load("rouge")

def evaluate_lstm_rouge(model, hf_dataset, n_samples=50, max_len=40):
    """Run greedy decoding on a subset, compute ROUGE + average overlap."""
    model.eval()
    preds = []
    refs = []
    overlaps = []

    # Create a DataLoader for the evaluation subset to handle batching
    eval_dataset = SummarizationSeq2SeqDataset(hf_dataset, stoi)
    eval_dataloader = DataLoader(eval_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

    with torch.no_grad():
        for batch in eval_dataloader:
            input_ids = batch["input_ids"].to(device)
            src_mask = batch["src_mask"].to(device)
            labels = batch["labels"]

            generated_ids_batch = model.generate(input_ids, src_mask, max_len=max_len)

            for i in range(len(input_ids)):
                pred_text = ids_to_text(generated_ids_batch[i])
                ref_text = ids_to_text(labels[i]) # Use the raw labels to reconstruct reference text

                preds.append(pred_text)
                refs.append(ref_text)
                overlaps.append(token_overlap(pred_text, ref_text))
                if len(preds) >= n_samples:
                    break
            if len(preds) >= n_samples:
                break

    rouge_scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    avg_overlap = sum(overlaps) / len(overlaps)

    return rouge_scores, avg_overlap, preds, refs


lstm_rouge, lstm_overlap, lstm_preds, lstm_refs = evaluate_lstm_rouge(model_lstm, val_small)

lstm_rouge, lstm_overlap

({'rouge1': np.float64(0.2905278371964319),
  'rouge2': np.float64(0.05808236273054479),
  'rougeL': np.float64(0.2771193205398773),
  'rougeLsum': np.float64(0.276745622118464)},
 0.15380913197569546)

In [ ]:
# === TODO === Qualitative LSTM inspection
for i in range(3):
    print("DIALOGUE:")
    print(val_small[i]["dialogue"])
    print("REFERENCE SUMMARY:")
    print(val_small[i]["summary"])
    print("LSTM PRED:")
    print(lstm_preds[i])
    print("=" * 80)

DIALOGUE:
A: Hi Tom, are you busy tomorrow’s afternoon?
B: I’m pretty sure I am. What’s up?
A: Can you go with me to the animal shelter?.
B: What do you want to do?
A: I want to get a puppy for my son.
B: That will make him so happy.
A: Yeah, we’ve discussed it many times. I think he’s ready now.
B: That’s good. Raising a dog is a tough issue. Like having a baby ;-) 
A: I'll get him one of those little dogs.
B: One that won't grow up too big;-)
A: And eat too much;-))
B: Do you know which one he would like?
A: Oh, yes, I took him there last Monday. He showed me one that he really liked.
B: I bet you had to drag him away.
A: He wanted to take it home right away ;-).
B: I wonder what he'll name it.
A: He said he’d name it after his dead hamster – Lemmy  - he's  a great Motorhead fan :-)))
REFERENCE SUMMARY:
A will go to the animal shelter tomorrow to get a puppy for her son. They already visited the shelter last Monday and the son chose the puppy. 
LSTM PRED:
<unk> <unk> <unk> <unk> <unk

## 3. Decoder-only Transformer: GPT2 (distilgpt2)
Now we fine-tune a small GPT2-like model (`distilgpt2`) for summarization.
Idea:
- Training input: a **single string** combining prompt + dialogue + summary:
- `"summarize the chat: <dialogue> \\n\\n summary: <summary>"`
- Objective: causal LM (next-token prediction).
- At test time:
- We give only `"summarize the chat: <dialogue> \\n\\n summary:"`
- Let the model generate the rest.
We keep training arguments minimal to avoid version conflicts with `TrainingArguments`.

In [ ]:

gpt2_model_name = "distilgpt2"
tokenizer_gpt2 = AutoTokenizer.from_pretrained(gpt2_model_name)
# GPT-2 has no pad token by default → reuse eos_token as pad
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token

def format_example_for_gpt2(ex):
    """Return a dict with a single "text" field built from dialogue + summary."""
    prefix = "summarize the chat: " + ex["dialogue"] + "\n\nsummary: "
    full = prefix + ex["summary"]
    return {"text": full, "prompt_len": len(tokenizer_gpt2(prefix, truncation=True, max_length=256).input_ids)}

# map format_example_for_gpt2 over train/val splits
train_gpt2 = train_small.map(format_example_for_gpt2, remove_columns=train_small.column_names)
val_gpt2 = val_small.map(format_example_for_gpt2, remove_columns=val_small.column_names)

def tokenize_for_gpt2(examples):
    """Tokenize the prompt text with truncation (max_length≈256)."""
    return tokenizer_gpt2(
        examples["text"],
        truncation=True,
        max_length=256,
    )

train_gpt2_tok = train_gpt2.map(tokenize_for_gpt2, batched=True)
val_gpt2_tok = val_gpt2.map(tokenize_for_gpt2, batched=True)

train_gpt2_tok.set_format("torch")
val_gpt2_tok.set_format("torch")

len(train_gpt2_tok), len(val_gpt2_tok)


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

(800, 100)

In [ ]:
# === TODO === Fine-tune GPT-2 with Trainer
model_gpt2 = AutoModelForCausalLM.from_pretrained(gpt2_model_name).to(device)
data_collator_gpt2 = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_gpt2,
    mlm=False,
)
training_args_gpt2 = TrainingArguments(
    output_dir="./gpt2_results",
    # evaluation_strategy="epoch", # Removed as it's causing TypeError and evaluation is manual
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./gpt2_logs",
    logging_steps=10,
    report_to="none", # Disable Weights & Biases or other external reporting
)

trainer_gpt2 = Trainer( # we won't rely on Trainer's eval metrics; we manually evaluate below
    model=model_gpt2,
    args=training_args_gpt2,
    train_dataset=train_gpt2_tok,
    eval_dataset=val_gpt2_tok,
    data_collator=data_collator_gpt2,
)

trainer_gpt2.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.552600
20,3.375200
30,3.355800
40,3.168700
50,3.221500
60,3.193700
70,3.101600
80,3.101000
90,3.085100
100,3.135500


TrainOutput(global_step=100, training_loss=3.2290651321411135, metrics={'train_runtime': 43.1892, 'train_samples_per_second': 18.523, 'train_steps_per_second': 2.315, 'total_flos': 50685445472256.0, 'train_loss': 3.2290651321411135, 'epoch': 1.0})

### 3.1 GPT2 generation and evaluation
We:
- Build the prompt `"summarize the chat: <dialogue> \\n\\nsummary:"`.
- Use `generate` with:
- beam search
- `no_repeat_ngram_size` to reduce repetition
- Compute ROUGE + token overlap on validation samples.

In [ ]:

# === TODO === GPT-2 generation + ROUGE evaluation helpers
def gpt2_generate_summary(dialogue, max_new_tokens=40):
    """Build the prompt, run model_gpt2.generate, decode, and strip the prompt prefix."""
    prompt = "summarize the chat: " + dialogue + "\n\nsummary:"
    input_ids = tokenizer_gpt2(prompt, return_tensors="pt").input_ids.to(device)

    # Calculate prompt length for stripping later
    prompt_len = input_ids.shape[1]

    # Generate summary with beam search
    output_ids = model_gpt2.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        num_beams=5,
        no_repeat_ngram_size=2,
        early_stopping=True,
        pad_token_id=tokenizer_gpt2.pad_token_id,
        eos_token_id=tokenizer_gpt2.eos_token_id
    )

    # Decode the generated tokens and strip the prompt
    generated_text = tokenizer_gpt2.decode(output_ids[0], skip_special_tokens=True)
    summary = generated_text[len(tokenizer_gpt2.decode(input_ids[0], skip_special_tokens=True)):].strip()

    # Ensure summary doesn't include remaining prompt if it hit max_new_tokens
    if "summarize the chat:" in summary:
        summary = summary.split("summarize the chat:")[0].strip()

    return summary

def evaluate_gpt2_rouge(hf_dataset, n_samples=50, max_new_tokens=40):
    """Loop over a subset, call gpt2_generate_summary, and compute ROUGE + overlap."""
    preds = []
    refs = []
    overlaps = []
    model_gpt2.eval()

    with torch.no_grad():
        for i in range(min(n_samples, len(hf_dataset))):
            example = hf_dataset[i]
            dialogue = example["dialogue"]
            reference_summary = example["summary"]

            predicted_summary = gpt2_generate_summary(dialogue, max_new_tokens=max_new_tokens)

            preds.append(predicted_summary)
            refs.append(reference_summary)
            overlaps.append(token_overlap(predicted_summary, reference_summary))

    rouge_scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    avg_overlap = sum(overlaps) / len(overlaps)
    return rouge_scores, avg_overlap, preds, refs

# assign outputs from evaluate_gpt2_rouge
gpt2_rouge, gpt2_overlap, gpt2_preds, gpt2_refs = evaluate_gpt2_rouge(
    val_small
)
gpt2_rouge, gpt2_overlap


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


({'rouge1': np.float64(0.2194551195220178),
  'rouge2': np.float64(0.03407038267688116),
  'rougeL': np.float64(0.15838716075232884),
  'rougeLsum': np.float64(0.162994784007169)},
 0.12662305901958817)

In [ ]:
# Inspect a few GPT2 summaries
for i in range(3):
    print("DIALOGUE:")
    print(val_small[i]["dialogue"])
    print("REFERENCE SUMMARY:")
    print(val_small[i]["summary"])
    print("GPT2 PRED:")
    print(gpt2_preds[i])
    print("=" * 80)

DIALOGUE:
A: Hi Tom, are you busy tomorrow’s afternoon?
B: I’m pretty sure I am. What’s up?
A: Can you go with me to the animal shelter?.
B: What do you want to do?
A: I want to get a puppy for my son.
B: That will make him so happy.
A: Yeah, we’ve discussed it many times. I think he’s ready now.
B: That’s good. Raising a dog is a tough issue. Like having a baby ;-) 
A: I'll get him one of those little dogs.
B: One that won't grow up too big;-)
A: And eat too much;-))
B: Do you know which one he would like?
A: Oh, yes, I took him there last Monday. He showed me one that he really liked.
B: I bet you had to drag him away.
A: He wanted to take it home right away ;-).
B: I wonder what he'll name it.
A: He said he’d name it after his dead hamster – Lemmy  - he's  a great Motorhead fan :-)))
REFERENCE SUMMARY:
A will go to the animal shelter tomorrow to get a puppy for her son. They already visited the shelter last Monday and the son chose the puppy. 
GPT2 PRED:
Tom and his dog will be euth

## 4. Encoder-decoder Transformer: T5-small
We fine-tune `t5-small` for summarization.
- Input: `"summarize: " + dialogue`
- Target: summary text
- Loss: cross-entropy on decoder tokens.
We use `Seq2SeqTrainer` but keep `TrainingArguments` simple to avoid version issues.
We will **not** rely on `predict_with_generate` in `TrainingArguments` - instead we
run generation manually for evaluation.

In [ ]:
# === TODO === T5 preprocessing + Trainer setup
t5_model_name = "t5-small"
tokenizer_t5 = AutoTokenizer.from_pretrained(t5_model_name)

max_input_length = 256
max_target_length = 64


def preprocess_t5(examples):
    """Return tokenized inputs ("summarize: dialogue") and target labels."""
    inputs = ["summarize: " + dialogue for dialogue in examples["dialogue"]]
    model_inputs = tokenizer_t5(
        inputs, max_length=max_input_length, truncation=True, padding="max_length"
    )

    # Setup the tokenizer for targets
    with tokenizer_t5.as_target_tokenizer():
        labels = tokenizer_t5(
            examples["summary"],
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_t5_tok = train_small.map(preprocess_t5, batched=True, remove_columns=train_small.column_names)
val_t5_tok = val_small.map(preprocess_t5, batched=True, remove_columns=val_small.column_names)

train_t5_tok.set_format("torch")
val_t5_tok.set_format("torch")

model_t5 = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name).to(device)
data_collator_t5 = DataCollatorForSeq2Seq(tokenizer_t5, model=model_t5)

training_args_t5 = Seq2SeqTrainingArguments(
    output_dir="./t5_results",
    # evaluation_strategy="epoch", # Removed as evaluation will be manual
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./t5_logs",
    logging_steps=10,
    report_to="none",
    predict_with_generate=False, # We will do generation manually
)

trainer_t5 = Seq2SeqTrainer(
    model=model_t5,
    args=training_args_t5,
    train_dataset=train_t5_tok,
    eval_dataset=val_t5_tok,
    tokenizer=tokenizer_t5,
    data_collator=data_collator_t5,
)

trainer_t5.train()


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/tmp/ipython-input-1391848274.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_t5 = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:740: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss
10,8.475200
20,7.120300
30,5.640500
40,4.908500
50,3.887500
60,3.302900
70,2.748000
80,2.405900
90,2.464600
100,2.404600


TrainOutput(global_step=100, training_loss=4.335795726776123, metrics={'train_runtime': 26.5644, 'train_samples_per_second': 30.116, 'train_steps_per_second': 3.764, 'total_flos': 54136720588800.0, 'train_loss': 4.335795726776123, 'epoch': 1.0})

### 4.1 T5 generation and evaluation
We again:
- Use `generate` with beam search.
- Compute ROUGE + token overlap.

In [ ]:

# === TODO === T5 generation + ROUGE evaluation helpers
def t5_generate_summary(dialogue, max_new_tokens=64):
    """Tokenize the prompt, run model_t5.generate, and decode the summary."""
    input_text = "summarize: " + dialogue
    input_ids = tokenizer_t5(input_text, return_tensors="pt", max_length=max_input_length, truncation=True).input_ids.to(device)

    output_ids = model_t5.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        num_beams=5,
        no_repeat_ngram_size=2,
        early_stopping=True,
        pad_token_id=tokenizer_t5.pad_token_id,
        eos_token_id=tokenizer_t5.eos_token_id
    )

    summary_text = tokenizer_t5.decode(output_ids[0], skip_special_tokens=True)
    return summary_text

def evaluate_t5_rouge(hf_dataset, n_samples=50, max_new_tokens=64):
    """Generate summaries for a subset and compute ROUGE + overlap."""
    preds = []
    refs = []
    overlaps = []
    model_t5.eval() # Set model to evaluation mode

    with torch.no_grad():
        for i in range(min(n_samples, len(hf_dataset))):
            example = hf_dataset[i]
            dialogue = example["dialogue"]
            reference_summary = example["summary"]

            predicted_summary = t5_generate_summary(dialogue, max_new_tokens=max_new_tokens)

            preds.append(predicted_summary)
            refs.append(reference_summary)
            overlaps.append(token_overlap(predicted_summary, reference_summary))

    rouge_scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    avg_overlap = sum(overlaps) / len(overlaps)

    return rouge_scores, avg_overlap, preds, refs

t5_rouge, t5_overlap, t5_preds, t5_refs = evaluate_t5_rouge(
    val_small
)
t5_rouge, t5_overlap


({'rouge1': np.float64(0.27546694001408384),
  'rouge2': np.float64(0.09201416890200328),
  'rougeL': np.float64(0.19644568830867887),
  'rougeLsum': np.float64(0.1959544374637054)},
 0.09248709675072797)

In [ ]:
for i in range(3):
    print("DIALOGUE:")
    print(val_small[i]["dialogue"])
    print("REFERENCE SUMMARY:")
    print(val_small[i]["summary"])
    print("T5 PRED:")
    print(t5_preds[i])
    print("=" * 80)


DIALOGUE:
A: Hi Tom, are you busy tomorrow’s afternoon?
B: I’m pretty sure I am. What’s up?
A: Can you go with me to the animal shelter?.
B: What do you want to do?
A: I want to get a puppy for my son.
B: That will make him so happy.
A: Yeah, we’ve discussed it many times. I think he’s ready now.
B: That’s good. Raising a dog is a tough issue. Like having a baby ;-) 
A: I'll get him one of those little dogs.
B: One that won't grow up too big;-)
A: And eat too much;-))
B: Do you know which one he would like?
A: Oh, yes, I took him there last Monday. He showed me one that he really liked.
B: I bet you had to drag him away.
A: He wanted to take it home right away ;-).
B: I wonder what he'll name it.
A: He said he’d name it after his dead hamster – Lemmy  - he's  a great Motorhead fan :-)))
REFERENCE SUMMARY:
A will go to the animal shelter tomorrow to get a puppy for her son. They already visited the shelter last Monday and the son chose the puppy. 
T5 PRED:
"i want to get a puppy for my 

## 5. Model comparison
We now compare:
- **LSTM seq2seq** (custom)
- **GPT2 decoder-only**
- **T5 encoder-decoder**
using ROUGE + token overlap.

In [ ]:
def pretty_print_rouge(name, scores, overlap):
    print(f"=== {name} ===")
    for k, v in scores.items():
        print(f"{k}: {float(v):.4f}")
    print(f"token_overlap: {overlap:.4f}")
    print()


pretty_print_rouge("LSTM seq2seq", lstm_rouge, lstm_overlap)
pretty_print_rouge("GPT2 decoder-only", gpt2_rouge, gpt2_overlap)
pretty_print_rouge("T5 encoder–decoder", t5_rouge, t5_overlap)

=== LSTM seq2seq ===
rouge1: 0.2905
rouge2: 0.0581
rougeL: 0.2771
rougeLsum: 0.2767
token_overlap: 0.1538

=== GPT2 decoder-only ===
rouge1: 0.2195
rouge2: 0.0341
rougeL: 0.1584
rougeLsum: 0.1630
token_overlap: 0.1266

=== T5 encoder–decoder ===
rouge1: 0.2755
rouge2: 0.0920
rougeL: 0.1964
rougeLsum: 0.1960
token_overlap: 0.0925



Looking ahead beyond this assignment, the most meaningful improvements would come from tightening control over preprocessing choices and treating generation behavior as part of the modeling problem. On the LSTM side, experimenting with vocabulary pruning thresholds, token normalization, or even a simple subword scheme would likely reduce the dominance of <unk> tokens and make summaries more informative. For transformer models, greater attention to decoding configuration and input framing can substantially improve faithfulness; several generated summaries drifted semantically despite reasonable ROUGE scores, highlighting the gap between automatic metrics and actual meaning preservation. More broadly, this assignment shows that strong pretrained models are only as reliable as the constraints placed around them. Developing an intuition for these tradeoffs is an important step toward building summarization systems that are not just fluent, but trustworthy.